# Exploración del pipeline de transacciones

El notebook recorre la lectura, limpieza, transformación y escritura del pipeline, mostrando el resultado de cada etapa.

**Hipótesis:** las transacciones con estado `APPROVED` pueden resumirse por fecha y BIN sin perder los totales de cantidad y monto aprobados.

In [1]:
from pathlib import Path
import os

project_root = Path.cwd().parent
os.chdir(project_root)

## 1. Lectura

Se carga el JSONL con la misma función utilizada por el pipeline. Se supone que cada línea representa una transacción y que `created_at` define su fecha.

In [2]:
import pandas as pd

from python.utils import (
    build_transaction_summary,
    clean_transactions,
    read_file,
    write_parquet,
)

input_path = os.path.join("data", "transactions_50k.jsonl")
output_path = os.path.join("output", "transactions_summary.parquet")
transactions = read_file(input_path)

La muestra omite el método de pago anidado para no visualizar información personal. En esta etapa se revisan tamaño, nulos, duplicados, periodo y estados antes de transformar los datos.

In [3]:
input_columns = [
    "id",
    "created_at",
    "updated_at",
    "status",
    "amount_in_cents",
]
input_preview = transactions.loc[:, input_columns].head()
input_preview

,id,created_at,updated_at,status,amount_in_cents
0,1720000000-77942,2024-07-03 16:12:37,2024-07-03 16:13:29,APPROVED,5155064
1,1720000010-45698,2024-04-01 10:40:39,2024-04-01 10:43:49,APPROVED,14700093
2,1720000020-64182,2024-09-11 18:12:59,2024-09-11 18:14:23,APPROVED,2143363
3,1720000030-25046,2024-09-03 17:15:37,2024-09-03 17:18:16,APPROVED,31278534
4,1720000040-39508,2024-06-15 10:54:17,2024-06-15 10:57:33,DECLINED,65863594


In [4]:
input_summary_data = {
    "metric": [
        "rows",
        "columns",
        "null_values",
        "duplicate_ids",
        "minimum_date",
        "maximum_date",
    ],
    "value": [
        len(transactions),
        len(transactions.columns),
        transactions.isna().sum().sum(),
        transactions["id"].duplicated().sum(),
        transactions["created_at"].min(),
        transactions["created_at"].max(),
    ],
}
input_summary = pd.DataFrame(input_summary_data)
input_summary

,metric,value
0,rows,50000
1,columns,6
2,null_values,0
3,duplicate_ids,0
4,minimum_date,2024-04-01 00:08:37
5,maximum_date,2024-09-28 23:54:45


In [5]:
status_summary = transactions["status"].value_counts().to_frame(
    name="transaction_count"
)
status_summary

,transaction_count
status,
APPROVED,42427
DECLINED,7031
ERROR,542


## 2. Limpieza

La limpieza conserva solo las columnas necesarias y aplica estas decisiones:

- `created_at` se convierte a fecha.
- `status` elimina espacios y se normaliza en mayúsculas.
- `amount_in_cents` se convierte a entero y se conserva en centavos para evitar pérdida de precisión.
- `bin` se extrae desde `payment_method_type.extra.bin`.
- No se imputan valores ni se eliminan filas de forma silenciosa.

In [6]:
cleaned_transactions = clean_transactions(transactions)
cleaned_transactions.head()

,id,created_at,status,amount_in_cents,bin
0,1720000000-77942,2024-07-03 16:12:37,APPROVED,5155064,450668
1,1720000010-45698,2024-04-01 10:40:39,APPROVED,14700093,512069
2,1720000020-64182,2024-09-11 18:12:59,APPROVED,2143363,438108
3,1720000030-25046,2024-09-03 17:15:37,APPROVED,31278534,459321
4,1720000040-39508,2024-06-15 10:54:17,DECLINED,65863594,546637


In [7]:
cleaning_summary_data = {
    "data_type": cleaned_transactions.dtypes.astype(str),
    "null_values": cleaned_transactions.isna().sum(),
    "unique_values": cleaned_transactions.nunique(dropna=False),
}
cleaning_summary = pd.DataFrame(cleaning_summary_data)
cleaning_summary

,data_type,null_values,unique_values
id,str,0,50000
created_at,datetime64[us],0,49924
status,string,0,3
amount_in_cents,int64,0,49959
bin,string,0,150


## 3. Transformación

Se consideran aprobadas únicamente las filas con estado `APPROVED`. La fecha se divide en día, mes y año, y después se agrupan los registros por esos campos y BIN para contar transacciones y sumar montos.

In [8]:
transaction_summary = build_transaction_summary(cleaned_transactions)
transaction_summary.head()

,transaction_date,month,year,bin,approved_transaction_count,approved_amount_in_cents
0,2024-04-01,4,2024,231030,1,28352749
1,2024-04-01,4,2024,232002,2,14537299
2,2024-04-01,4,2024,238000,2,15865527
3,2024-04-01,4,2024,400489,1,2499553
4,2024-04-01,4,2024,400490,1,758826530


In [9]:
group_columns = ["transaction_date", "month", "year", "bin"]
transformation_summary_data = {
    "metric": [
        "summary_rows",
        "approved_transactions",
        "approved_amount_in_cents",
        "duplicate_keys",
    ],
    "value": [
        len(transaction_summary),
        transaction_summary["approved_transaction_count"].sum(),
        transaction_summary["approved_amount_in_cents"].sum(),
        transaction_summary.duplicated(group_columns).sum(),
    ],
}
transformation_summary = pd.DataFrame(transformation_summary_data)
transformation_summary

,metric,value
0,summary_rows,21529
1,approved_transactions,42427
2,approved_amount_in_cents,1965579524955
3,duplicate_keys,0


## 4. Salida

La vista agregada se escribe en Parquet porque conserva tipos y ofrece almacenamiento columnar. Se usa una ruta estable para que cada ejecución reemplace el mismo resultado y mantenga la idempotencia. Después se vuelve a leer el archivo para compararlo con la transformación en memoria.

In [10]:
write_parquet(transaction_summary, output_path)
parquet_output = pd.read_parquet(output_path, engine="pyarrow")
parquet_output.head()

,transaction_date,month,year,bin,approved_transaction_count,approved_amount_in_cents
0,2024-04-01,4,2024,231030,1,28352749
1,2024-04-01,4,2024,232002,2,14537299
2,2024-04-01,4,2024,238000,2,15865527
3,2024-04-01,4,2024,400489,1,2499553
4,2024-04-01,4,2024,400490,1,758826530


In [11]:
output_summary_data = {
    "metric": [
        "rows",
        "columns",
        "null_values",
        "matches_transformation",
    ],
    "value": [
        len(parquet_output),
        len(parquet_output.columns),
        parquet_output.isna().sum().sum(),
        parquet_output.equals(transaction_summary),
    ],
}
output_summary = pd.DataFrame(output_summary_data)
output_summary

,metric,value
0,rows,21529
1,columns,6
2,null_values,0
3,matches_transformation,True


## Conclusiones

- Se leen 50.000 transacciones sin identificadores duplicados.
- La limpieza conserva únicamente los campos requeridos para el reto.
- La transformación produce 21.529 filas y 42.427 transacciones aprobadas.
- El contenido persistido en Parquet coincide con la transformación en memoria.
- La hipótesis se cumple: la agregación conserva los totales aprobados sin claves duplicadas.